In [1]:
from pairs_trading.data.loaders import load_prices
from pairs_trading.stats.cointegration import EGResult, JohansenResult
from pairs_trading.config import SplitConfig

Prepare dictionary of tickers to test for cointegration

In [2]:
ticker_dict = {
    'metals':["GLD", "IAU", "SLV", "GDX"],
    "US":["SPY", "IVV", "VTI"],
    "fixed":["TLT", "IEF", "AGG"],
    "energy":["XLE", "XOP"],
    "finance": ["XLF", "KBE"], 
    "real_estate": ["VNQ", "REM"]
    }

Create the full df with all tickers in a short (6 month) time frame to ensure that structural drifts do not wash out cointegration

In [3]:
all_tickers = [ticker for tickers in ticker_dict.values() for ticker in tickers]
df = load_prices(all_tickers, SplitConfig.coint_start, SplitConfig.coint_end, refresh=True)

[*********************100%***********************]  16 of 16 completed


### Egle-Granger Tests

In [4]:
from itertools import combinations

for key in ticker_dict.keys():
    pairs = list(combinations(ticker_dict[key], 2))
    print(f"Results for {key}:")
    for pair in pairs:
        result = EGResult(df, pair)
        result.print_results()
    print("-"*40)

Results for metals:
Engle-Granger Result: y = IAU, x=GLD (statistic=-5.20925320214552, p-value=6.666797894165647e-05, cointegrated: True)
Engle-Granger Result: y = SLV, x=GLD (statistic=-4.269748764545141, p-value=0.0028528150979858675, cointegrated: True)
Engle-Granger Result: y = GLD, x=GDX (statistic=-1.6655482642143515, p-value=0.692447995734903, cointegrated: False)
Engle-Granger Result: y = SLV, x=IAU (statistic=-4.264617547756867, p-value=0.002905360231813596, cointegrated: True)
Engle-Granger Result: y = IAU, x=GDX (statistic=-1.6562978875397099, p-value=0.696553530373216, cointegrated: False)
Engle-Granger Result: y = SLV, x=GDX (statistic=-1.6312817571809564, p-value=0.7075073902448366, cointegrated: False)
----------------------------------------
Results for US:
Engle-Granger Result: y = IVV, x=SPY (statistic=-10.397034103597216, p-value=2.5116772163267053e-17, cointegrated: True)
Engle-Granger Result: y = VTI, x=SPY (statistic=-2.0993633669422325, p-value=0.4766999702243756

### Johansen Test

In [5]:
from itertools import combinations

for key in ticker_dict.keys():
    pairs = list(combinations(ticker_dict[key], 2))
    print(f"Results for {key}:")
    for pair in pairs:
        result = JohansenResult(df, pair)
        print(f"{pair[0]}, {pair[1]}:")
        result.print_results()
    print("-"*40)

Results for metals:
GLD, IAU:
Johansen Result: (statistic: 31.864477894416876, critical values:15.4943, cointegrated: True)
GLD, SLV:
Johansen Result: (statistic: 16.437425971653507, critical values:15.4943, cointegrated: True)
GLD, GDX:
Johansen Result: (statistic: 14.155536125770592, critical values:15.4943, cointegrated: False)
IAU, SLV:
Johansen Result: (statistic: 16.185427414956475, critical values:15.4943, cointegrated: True)
IAU, GDX:
Johansen Result: (statistic: 14.347580169261212, critical values:15.4943, cointegrated: False)
SLV, GDX:
Johansen Result: (statistic: 20.31854717705056, critical values:15.4943, cointegrated: True)
----------------------------------------
Results for US:
SPY, IVV:
Johansen Result: (statistic: 46.87647442153707, critical values:15.4943, cointegrated: True)
SPY, VTI:
Johansen Result: (statistic: 7.911119961839432, critical values:15.4943, cointegrated: False)
IVV, VTI:
Johansen Result: (statistic: 7.7780893101099355, critical values:15.4943, cointeg

Based on these results I will select pairs that pass one of the tests (or are very close as for IAU/GDX) and are not redundant or trivial (as the many metal combinations). Hence, I pick:

- IAU/GDX
- GLD/SLV
- XLF/KBE
- SPY/IVV (follows the same index so profit should be arbitraged away but can be used as a test)
